# 05 Real Strategy Integration Example
Question: What does the end-to-end Part II path look like when real Part I macro files are paired with the current strategy return input?

**Warning:** This notebook is a research-only integration example and must not be used for forecasting, signal generation, optimization, allocation, or trading decisions.

This notebook demonstrates the first real macro handoff path in Part II: load the actual Part I regime and transition files, pair them with the current strategy return input, run the locked validations, and report the coverage boundary clearly.


## Load Real Macro Inputs And Validate The Contract
Use the checked-in real Part I macro files and the current strategy return input. Validation is required before any integration summary or joined example is treated as in scope.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.conditioning.regime_join import join_returns_with_regimes
from src.data.integration_summary import (
    build_macro_regime_category_counts,
    build_real_macro_integration_overview,
)
from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

run_all_validations(regimes, transitions, strategy_returns)

integration_overview, coverage = build_real_macro_integration_overview(
    regimes,
    transitions,
    strategy_returns,
)
display(integration_overview)


## Transition Coverage Interpretation
Zero aligned transitions is acceptable in this example when the sample strategy date range does not include any transition months. That is a strategy-coverage limitation, not a macro contract failure.


In [ ]:
transition_coverage_summary = integration_overview[
    [
        "transitions_inside_strategy_coverage",
        "transitions_outside_strategy_coverage",
    ]
]
display(transition_coverage_summary)

coverage_note = (
    "Zero aligned transitions is acceptable for this sample strategy range."
    if len(coverage.aligned) == 0
    else "Some transition months are inside the sample strategy coverage window."
)
display({"coverage_note": coverage_note})


## Joined Strategy Returns With Realized Macro Labels
This is the simplest end-to-end integration table: the current strategy returns joined to the real Part I macro regime labels by exact month-end `DATE`.


In [ ]:
joined = join_returns_with_regimes(strategy_returns, regimes)
joined_preview = joined[
    [
        "DATE",
        "strategy_id",
        "returns",
        "Growth_Regime",
        "Inflation_Regime",
        "Policy_Regime",
        "Macro_Regime",
    ]
].sort_values(["strategy_id", "DATE"]).reset_index(drop=True)
display(joined_preview)


## Realized Macro Regime Categories
Show the realized macro label frequencies from the real Part I regime file so the integration example remains table-oriented and easy to review.


In [ ]:
macro_regime_category_counts = build_macro_regime_category_counts(regimes)
display(macro_regime_category_counts)
